In [5]:
# Packages
import os
import pandas as pd
from pathlib import Path

# Directory
directory = r"C:\Users\20234503\Desktop\Research\Weather and Strikes\Project\Results_2"

# Strikes (No Duplicates within County)
strikes_baseline_nodup = pd.read_csv(os.path.join(directory, "Raw", "baseline_strike_nodup.csv"))
strikes_clim1_nodup    = pd.read_csv(os.path.join(directory, "Raw", "cdd_1_strike_nodup.csv"))  # e.g. Cold
strikes_clim2_nodup    = pd.read_csv(os.path.join(directory, "Raw", "cdd_2_strike_nodup.csv"))  # e.g. Hot

# Strikes (Weighted Sample)
strikes_baseline_weighted = pd.read_csv(os.path.join(directory, "Raw", "baseline_strike_weighted.csv"))
strikes_clim1_weighted    = pd.read_csv(os.path.join(directory, "Raw", "cdd_1_strike_weighted.csv"))
strikes_clim2_weighted    = pd.read_csv(os.path.join(directory, "Raw", "cdd_2_strike_weighted.csv"))

def clean_df(df):
    # Rename columns from the esttab-style export
    df = df.rename(columns={
        "Unnamed: 0": "Variable",
        ".": "est",
        "Unnamed: 2": "se"
    })
    
    # Try to select the 'Heat Day' row if it exists
    if "Variable" in df.columns and "Heat Day" in df["Variable"].values:
        df = df[df["Variable"] == "Heat Day"].reset_index(drop=True)
    else:
        # Fallback: take the third row if it exists, otherwise the last row
        if len(df) > 2:
            df = df.iloc[[2]].reset_index(drop=True)
        else:
            df = df.tail(1).reset_index(drop=True)
    
    return df

# Clean all the dataframes
strikes_baseline_nodup    = clean_df(strikes_baseline_nodup)
strikes_clim1_nodup       = clean_df(strikes_clim1_nodup)
strikes_clim2_nodup       = clean_df(strikes_clim2_nodup)

strikes_baseline_weighted = clean_df(strikes_baseline_weighted)
strikes_clim1_weighted    = clean_df(strikes_clim1_weighted)
strikes_clim2_weighted    = clean_df(strikes_clim2_weighted)

# #Obs (replace if you have spec-specific counts)
obs_baseline = 71024
obs_clim1    = 40112   # Cold region
obs_clim2    = 30912   # Hot region

def starify_from_t(t):
    """Return LaTeX stars from a t-stat."""
    t_abs = abs(t)
    if t_abs >= 2.58:
        return r"\sym{***}"
    elif t_abs >= 1.96:
        return r"\sym{**}"
    elif t_abs >= 1.64:
        return r"\sym{*}"
    else:
        return ""

def fmt_coef(est, se):
    """coef (3dp) + stars, using t = est/se."""
    t = est / se if se != 0 else 0
    return f"{est:.3f}{starify_from_t(t)}"

def fmt_se(se):
    """Standard error in parentheses, 3dp."""
    return f"({se:.3f})"

lines = []

lines.append(r"\def\sym#1{\ifmmode^{#1}\else\(^{#1}\)\fi}")
# 1 label column + 4 numeric columns
lines.append(r"\begin{tabular}{l*{4}{c}}")
lines.append(r"\toprule")
lines.append(
    r"                    &\multicolumn{2}{c}{(1)}&\multicolumn{2}{c}{(2)}\\"
)
lines.append(
    r"                    &\multicolumn{2}{c}{County-Collapsed}&\multicolumn{2}{c}{1/k Weights}\\"
)
lines.append(r"\midrule")

# =======================
# Panel A: All summer days
# =======================
lines.append(r"\multicolumn{5}{l}{\textbf{Panel A:} All Summer Days}\\")

# Extract baseline estimates
est_A_nodup = float(strikes_baseline_nodup.loc[0, "est"])
se_A_nodup  = float(strikes_baseline_nodup.loc[0, "se"])

est_A_w     = float(strikes_baseline_weighted.loc[0, "est"])
se_A_w      = float(strikes_baseline_weighted.loc[0, "se"])

coef_cells_A = [
    rf"\multicolumn{{2}}{{c}}{{{fmt_coef(est_A_nodup, se_A_nodup)}}}",
    rf"\multicolumn{{2}}{{c}}{{{fmt_coef(est_A_w, se_A_w)}}}",
]
se_cells_A = [
    rf"\multicolumn{{2}}{{c}}{{{fmt_se(se_A_nodup)}}}",
    rf"\multicolumn{{2}}{{c}}{{{fmt_se(se_A_w)}}}",
]

lines.append("Heat Day             &" + " & ".join(coef_cells_A) + r"\\")
lines.append("                    &" + " & ".join(se_cells_A) + r"\\")

# Panel A # Obs
lines.append(r"\midrule")
obs_cells_A = [
    rf"\multicolumn{{2}}{{c}}{{{obs_baseline}}}",
    rf"\multicolumn{{2}}{{c}}{{{obs_baseline}}}",
]
lines.append(r" \# Obs.            &" + " & ".join(obs_cells_A) + r"\\")
lines.append(r"\midrule")

# =======================
# Panel B: by climate region
# =======================
lines.append(r"\multicolumn{5}{l}{\textbf{Panel B:} Effect by Hot and Cold Climate Regions}\\")
lines.append(
    r"                    &\multicolumn{2}{c}{County-Collapsed}&\multicolumn{2}{c}{1/k Weights}\\"
)
lines.append(
    r"                    &\multicolumn{1}{c}{Cold}&\multicolumn{1}{c}{Hot}&"
    r"\multicolumn{1}{c}{Cold}&\multicolumn{1}{c}{Hot}\\"
)

# Estimates for climate regions
est_cold_nodup = float(strikes_clim1_nodup.loc[0, "est"])
se_cold_nodup  = float(strikes_clim1_nodup.loc[0, "se"])

est_hot_nodup = float(strikes_clim2_nodup.loc[0, "est"])
se_hot_nodup  = float(strikes_clim2_nodup.loc[0, "se"])

est_cold_w = float(strikes_clim1_weighted.loc[0, "est"])
se_cold_w  = float(strikes_clim1_weighted.loc[0, "se"])

est_hot_w = float(strikes_clim2_weighted.loc[0, "est"])
se_hot_w  = float(strikes_clim2_weighted.loc[0, "se"])

coef_cells_B = [
    fmt_coef(est_cold_nodup, se_cold_nodup),
    fmt_coef(est_hot_nodup,  se_hot_nodup),
    fmt_coef(est_cold_w,     se_cold_w),
    fmt_coef(est_hot_w,      se_hot_w),
]

se_cells_B = [
    fmt_se(se_cold_nodup),
    fmt_se(se_hot_nodup),
    fmt_se(se_cold_w),
    fmt_se(se_hot_w),
]

lines.append("Heat Day             &" + " & ".join(coef_cells_B) + r"\\")
lines.append("                    &" + " & ".join(se_cells_B) + r"\\")

# Panel B # Obs: 4 separate cells (same numbers used for both specs here)
lines.append(r"\midrule")
obs_cells_B = [
    str(obs_clim1),  # County-Collapsed, Cold
    str(obs_clim2),  # County-Collapsed, Hot
    str(obs_clim1),  # 1/k Weights, Cold
    str(obs_clim2),  # 1/k Weights, Hot
]
lines.append(r" \# Obs.            &" + " & ".join(obs_cells_B) + r"\\")

# FE rows
lines.append(
    r"\midrule "
    r"County, Month, Year FE & \multicolumn{2}{c}{\checkmark}& \multicolumn{2}{c}{\checkmark} \\"
    r"Con. Temp. & \multicolumn{2}{c}{\checkmark} & \multicolumn{2}{c}{\checkmark} \\"
)

lines.append(r"\bottomrule \addlinespace")
lines.append(r"\end{tabular}")

latex_table = "\n".join(lines)

# Write to .tex file
out_path = Path(directory) / "table_results_1_alt_sample.tex"
out_path.write_text(latex_table, encoding="utf-8")
print(f"Wrote LaTeX table to {out_path.resolve()}")


Wrote LaTeX table to C:\Users\20234503\Desktop\Research\Weather and Strikes\Project\Results_2\table_results_1_alt_sample.tex
